### **DimUsers**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

import os
import sys

project_pth = os.path.join(os.getcwd(),"..","..")
sys.path.append(project_pth)

from utils.transformations import reusable

### **Auto Loader**

In [0]:
df_user = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format","parquet")\
                .option("cloudFiles.schemaLocation","abfss://silver@storageazureprojectde.dfs.core.windows.net/DimUser/chekpoint")\
                .load("abfss://bronze@storageazureprojectde.dfs.core.windows.net/DimUser")

In [0]:
# display(df_user)

In [0]:
# basic transformations
df_user_obj = reusable()
df_user = df_user.withColumn("user_name",upper(col("user_name")))
df_user = df_user_obj.dropColumns(df_user, ['_rescued_data' ])
df_user = df_user.dropDuplicates(['user_id'])

In [0]:

df_user.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","abfss://silver@storageazureprojectde.dfs.core.windows.net/DimUser/chekpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageazureprojectde.dfs.core.windows.net/DimUser/data")\
    .toTable("spotify_cata.silver.DimUser")


### **DimArtists**

In [0]:
df_art = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format","parquet")\
                .option("cloudFiles.schemaLocation","abfss://silver@storageazureprojectde.dfs.core.windows.net/DimArtist/checkpoint")\
                .load("abfss://bronze@storageazureprojectde.dfs.core.windows.net/DimArtist")

In [0]:
# display(df_art)

In [0]:
# basic transformations
df_art_obj = reusable()
df_art = df_art_obj.dropColumns(df_art, ['_rescued_data' ])
df_art = df_art.dropDuplicates(['artist_id'])

In [0]:

df_art.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","abfss://silver@storageazureprojectde.dfs.core.windows.net/DimArtist/chekpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageazureprojectde.dfs.core.windows.net/DimArtist/data")\
    .toTable("spotify_cata.silver.DimArtist")


### **DimTrack**

In [0]:
df_track = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format","parquet")\
                .option("cloudFiles.schemaLocation","abfss://silver@storageazureprojectde.dfs.core.windows.net/DimTrack/checkpoint")\
                .load("abfss://bronze@storageazureprojectde.dfs.core.windows.net/DimTrack")

In [0]:
# duration flag categorization

df_track = df_track.withColumn("durationFlag",when(col('duration_sec')<150,"low")\
                                            .when(col('duration_sec')<300,"medium")\
                                            .otherwise("high"))

# string modification
df_track = df_track.withColumn("track_name",regexp_replace(col('track_name'),'-',' '))

#column drop
df_track = reusable().dropColumns(df_track, ['_rescued_data' ])

In [0]:

df_track.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","abfss://silver@storageazureprojectde.dfs.core.windows.net/DimTrack/chekpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageazureprojectde.dfs.core.windows.net/DimTrack/data")\
    .toTable("spotify_cata.silver.DimTrack")


### **DimDate**

In [0]:
df_date = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format","parquet")\
                .option("cloudFiles.schemaLocation","abfss://silver@storageazureprojectde.dfs.core.windows.net/DimDate/checkpoint")\
                .load("abfss://bronze@storageazureprojectde.dfs.core.windows.net/DimDate")

In [0]:
#drop column
df_date = reusable().dropColumns(df_date,['_rescued_data'])

#write data
df_date.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","abfss://silver@storageazureprojectde.dfs.core.windows.net/DimDate/chekpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageazureprojectde.dfs.core.windows.net/DimDate/data")\
    .toTable("spotify_cata.silver.DimDate")



### **FactStream**

In [0]:
df_fact = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format","parquet")\
                .option("cloudFiles.schemaLocation","abfss://silver@storageazureprojectde.dfs.core.windows.net/FactStream/checkpoint")\
                .load("abfss://bronze@storageazureprojectde.dfs.core.windows.net/FactStream")

In [0]:
df_fact = reusable().dropColumns(df_fact,['_rescued_data'])


df_fact.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","abfss://silver@storageazureprojectde.dfs.core.windows.net/FactStream/chekpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageazureprojectde.dfs.core.windows.net/FactStream/data")\
    .toTable("spotify_cata.silver.FactStream")